<a href="https://colab.research.google.com/github/ajou-cf/cloudtrail-attack-gliclass/blob/notebook%2Ffinetuning/notebooks/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. GLiClass 퓨샷 파인튜닝 (훈련 / 검증 / 시험)

**목적**: 04에서 나눈 데이터로 GLiClass edge 모델을 퓨샷 파인튜닝하고, 검증 세트로 에폭을 고른 뒤 시험 세트에서 한 번만 평가한다. reference의 시범 코드(`reference/pilot1.ipynb`, `pilot2.ipynb`)를 바탕으로, 검증하면서 찾은 문제를 고쳤다.

**입력** (04 결과, 드라이브 `splits/{A|B}/`)
- `kshot/train_k{k}_s{seed}.json`: 학습 파일 (라벨마다 k개, 후보 라벨 = 학습 라벨 5개)
- `val_gliclass.json`, `val.jsonl`: 검증 (후보 라벨 5개). 에폭·학습률 고르기에만 쓴다
- `test_gliclass.json`, `test.jsonl`: 시험 (후보 라벨 = 학습 5 + 숨긴 3). 마지막에 한 번만 쓴다
- `labels.csv`: 라벨 문장, 학습/숨김 구분

**출력** (드라이브 `results/finetune_{A|B}/`)
- `runs.csv`: 실행(k, seed)마다 고른 에폭과 시험 점수
- `curves.csv`: 에폭마다 학습 손실, 검증 손실, 검증 F1
- `per_label.csv`: 라벨별 시험 점수
- 그래프 4개 (`fig_*.png`)

**reference 시범 코드에서 고친 점**

| # | 시범 코드 | 문제 | 이 노트북 |
|---|---|---|---|
| 1 | `--val_data_path`에 학습 파일을 넣고 `save_steps`를 크게 해서 검증을 끔 | 검증 세트가 없어 에폭·학습률을 고를 근거가 없음. 시험 점수로 고르면 시험 누수 | 04의 `val_gliclass.json`으로 에폭마다 검증, 가장 좋은 에폭을 고른 뒤 시험은 한 번만 |
| 2 | seed는 예시 뽑기에만 씀 | `train.py`에 시드 설정이 없어 같은 seed로 다시 돌려도 결과가 달라짐 | `train.py`를 감싸서 학습 전에 `random`, `numpy`, `torch` 시드를 고정 |
| 3 | 1등 라벨 하나만 정답과 비교 (단일 라벨) | 우리 윈도우는 라벨이 2개 붙기도 함 (예: Discovery + Initial Access) | 라벨마다 임계값으로 판정하는 멀티라벨 지표 (micro/macro F1, 라벨별 F1, PR-AUC) + 1등 정확도도 같이 |
| 4 | 손실을 출력 문자열에서 정규식으로 뽑음 | 형식이 바뀌면 조용히 NaN | 체크포인트의 `trainer_state.json` 기록에서 읽음 |
| 5 | `train.py`가 출력하는 accuracy/F1을 볼 수 있음 | 라벨 칸을 전부 펼쳐 weighted 평균을 내서, 대부분 0인 칸 덕분에 항상 높게 나옴 | 무시하고 우리 지표로 평가 (검증 손실만 사용) |

**준비**: 런타임 유형 **T4 GPU**. 01~04를 돌려서 드라이브에 `splits/`가 있어야 함. 처음에는 `SMOKE = True`로 몇 분 안에 끝까지 도는지 먼저 확인한다.

**칸마다 보는 법**: `[하는 일]` / `[볼 것]` / `[확인 질문]` (02~04와 같음)

## 1. 설치

[하는 일] GLiClass를 시범 코드와 같은 커밋으로 설치하고, 학습 스크립트 `train.py`가 있는 저장소를 받는다. 이미 있으면 건너뛴다.

설치 뒤 다음 칸에서 import 에러가 나면: 런타임 > 세션 다시 시작 후 **2번부터** 다시 실행.

In [ ]:
import os, sys, subprocess, importlib.util

COMMIT = '6a00f8067648f25aa0f4f477d167c5f391cb4165'   # reference 시범 코드와 같은 버전

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive', '/content/work'
except ImportError:                                   # Colab이 아닌 곳: DATA_DIR(드라이브 대신 쓸 폴더)을 환경변수로
    ROOT = os.environ.get('DATA_DIR', '.')
    WORK = os.environ.get('WORK_DIR', os.path.join(ROOT, 'finetune_work'))
os.makedirs(WORK, exist_ok=True)

need = [p for p, m in [(f'git+https://github.com/Knowledgator/GLiClass.git@{COMMIT}', 'gliclass'),
                       ('datasets', 'datasets'), ('accelerate', 'accelerate')] if importlib.util.find_spec(m) is None]
if need:                                              # accelerate: train.py의 Trainer가 필요로 함
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
REPO = os.path.join(WORK, 'gliclass_repo')
if not os.path.exists(f'{REPO}/train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Knowledgator/GLiClass.git', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', COMMIT], check=True)
print('train.py:', os.path.exists(f'{REPO}/train.py'), '| WORK =', WORK)

## 2. 설정

[하는 일] 실험 조건을 한 곳에 모은다. 바꿔볼 값은 여기만 고치면 된다.

- `VARIANT`: 04의 A(압축 안 함) 또는 B(반복 압축)
- `EPOCHS`: 최대 에폭. 에폭마다 체크포인트를 남기고 검증으로 가장 좋은 에폭을 고른다
- `SELECT_BY`: 에폭을 고르는 검증 지표
- `SMOKE = True`: k=1, seed 0, 2에폭, 평가 일부만. 끝까지 도는지 확인용 (T4에서 약 5분)

In [ ]:
import json, math, glob, shutil, time, random
import numpy as np, pandas as pd, torch

VARIANT = 'B'
MODEL = 'knowledgator/gliclass-edge-v3.0'
K_LIST = [1, 3, 5, 8, 16]
SEEDS = [0, 1, 2]
EPOCHS = 10
ENCODER_LR = 2e-5          # train.py 기본값
OTHERS_LR = 2e-5           # train.py 기본값은 5e-5. 시범 코드는 인코더와 같게 둠
BATCH = 8
MAX_LEN = 1024             # 4번에서 실제 토큰 수로 확인
FOCAL_ALPHA, FOCAL_GAMMA = 0.5, 2.0
THRESHOLD = 0.5            # 멀티라벨 판정 임계값
SELECT_BY = 'val_macro_f1'
TRAINER_EVAL_N = 500       # train.py가 에폭마다 검증 손실을 낼 때 쓰는 검증 개수
AGNEWS_N = 500             # 원래 실력 확인용 ag_news 개수
KEEP_MODELS = False        # True면 고른 에폭의 모델을 WORK에 남김
EXTRA_ARGS = []            # train.py에 더 넘길 인자. 예: ['--use_ewc', '--prev_data_path', 'prev.json']

SMOKE = False
VAL_N = TEST_N = None      # None이면 전부
if SMOKE:
    K_LIST, SEEDS, EPOCHS = [1], [0], 2
    VAL_N, TEST_N, AGNEWS_N, TRAINER_EVAL_N = 200, 300, 100, 100

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (CPU로는 매우 느림)')

## 3. 데이터 불러오기 (04 결과)

[하는 일] 드라이브에서 `splits/{VARIANT}`와 `labels.csv`를 찾아 읽는다.

[볼 것] pool/val/test 개수가 04의 마지막 표와 같은지.

In [ ]:
split_hits = glob.glob(f'{ROOT}/**/splits/{VARIANT}/val_gliclass.json', recursive=True)
lab_hits = glob.glob(f'{ROOT}/**/labels.csv', recursive=True)
assert split_hits and lab_hits, f'splits/{VARIANT} 또는 labels.csv를 못 찾음 (04 먼저)'
SPLIT = os.path.dirname(split_hits[0])
DATA_DIR = os.path.dirname(lab_hits[0])
OUT = f'{DATA_DIR}/results/finetune_{VARIANT}' + ('_smoke' if SMOKE else '')
os.makedirs(OUT, exist_ok=True)

labels = pd.read_csv(lab_hits[0])
TRAIN = labels[labels.role == 'train'].label_id.tolist()
HELD = labels[labels.role == 'held_out'].label_id.tolist()
ATTACK_TRAIN = [l for l in TRAIN if l != 'BENIGN']
name = dict(zip(labels.label_id, labels.name))
text_of = dict(zip(labels.label_id, labels.label_text))
id_of = {t: l for l, t in text_of.items()}

parts = {sp: pd.read_json(f'{SPLIT}/{sp}.jsonl', lines=True) for sp in ['pool', 'val', 'test']}
gl = {sp: json.load(open(f'{SPLIT}/{sp}_gliclass.json')) for sp in ['val', 'test']}
train_files = {}
for f in sorted(glob.glob(f'{SPLIT}/kshot/train_k*_s*.json')):
    k, s = map(int, os.path.basename(f)[7:-5].split('_s'))
    train_files[(k, s)] = f

print('SPLIT =', SPLIT, '| 결과 =', OUT)
print({sp: len(d) for sp, d in parts.items()})
print('학습 라벨:', TRAIN, '| 숨긴 라벨:', HELD)
print('k-shot 파일:', sorted(train_files))

## 4. 데이터 검사

[하는 일] 학습 전에 데이터가 규칙대로인지 확인한다. 하나라도 틀리면 멈춘다.

1. 평가 파일(gliclass 형식)과 jsonl의 개수·문장이 같은지
2. 학습·검증 파일의 후보 라벨이 학습 라벨 5개뿐인지 (숨긴 라벨 문장이 섞이면 제로샷 실험이 깨짐)
3. 시험 파일의 후보 라벨이 8개(학습 5 + 숨김 3)인지
4. 학습 문장이 검증·시험에 없는지 (누수)
5. 파일 안의 라벨 문장이 지금 `labels.csv`와 같은지 (labels.csv를 고친 뒤 04를 안 돌린 경우)
6. k-shot 파일마다 라벨별 개수가 k 이상인지
7. 토큰 길이가 `MAX_LEN` 안에 드는지

[볼 것] 마지막 줄에 `검사 통과`.

In [ ]:
from transformers import AutoTokenizer

for sp in ['val', 'test']:
    assert len(gl[sp]) == len(parts[sp]), f'{sp}: gliclass {len(gl[sp])} vs jsonl {len(parts[sp])}'
    assert [r['text'] for r in gl[sp]] == parts[sp].text.tolist(), f'{sp}: 문장 순서 다름'

train_texts = {text_of[l] for l in TRAIN}
assert all(set(r['all_labels']) == train_texts for r in gl['val']), 'val 후보 라벨이 학습 라벨 5개가 아님'
assert all(set(r['all_labels']) == {text_of[l] for l in TRAIN + HELD} for r in gl['test']), 'test 후보 라벨이 8개가 아님'

seen_eval = set(parts['val'].text) | set(parts['test'].text)
rows = []
for (k, s), f in sorted(train_files.items()):
    tr = json.load(open(f))
    assert all(set(r['all_labels']) == train_texts for r in tr), f'{f}: 후보 라벨에 숨긴 라벨'
    assert not ({r['text'] for r in tr} & seen_eval), f'{f}: 학습 문장이 검증/시험에 있음'
    assert all(t in id_of for r in tr for t in r['true_labels']), f'{f}: labels.csv에 없는 라벨 문장 (04 다시)'
    cnt = pd.Series([id_of[t] for r in tr for t in r['true_labels']]).value_counts()
    assert all(cnt.get(l, 0) >= k for l in TRAIN), f'{f}: 라벨별 {k}개 미만'
    rows.append({'k': k, 'seed': s, 'windows': len(tr), **{name[l]: int(cnt.get(l, 0)) for l in TRAIN}})
print(pd.DataFrame(rows).to_string(index=False))

tok = AutoTokenizer.from_pretrained(MODEL)
lab_tok = sum(len(tok(text_of[l], add_special_tokens=False)['input_ids']) + 1 for l in TRAIN + HELD)
txt_tok = pd.concat([parts[sp].text for sp in parts]).map(lambda t: len(tok(t)['input_ids']))
print(f'\n토큰: 문장 평균 {txt_tok.mean():.0f} / 최대 {txt_tok.max()} + 라벨 8개 {lab_tok} = 최대 {txt_tok.max() + lab_tok} (MAX_LEN {MAX_LEN})')
assert txt_tok.max() + lab_tok <= MAX_LEN, 'MAX_LEN을 늘리기 (넘치면 문장 뒤쪽이 잘림)'
print('검사 통과')

## 5. 평가 함수

[하는 일] 모델 하나를 받아 라벨마다 sigmoid 점수를 내고, 여러 지표를 계산한다.

| 이름 | 데이터 / 후보 라벨 | 뜻 |
|---|---|---|
| `val_*` | 검증 / 학습 5개 | 에폭 고르기용 |
| `seen_*` | 시험 중 숨긴 라벨 없는 윈도우 / 학습 5개 | 학습한 전술 성능 |
| `held_*` | 시험 중 숨긴 라벨 윈도우 / 숨긴 3개 + BENIGN | 처음 보는 전술 (제로샷). 학습 라벨과 경쟁시키지 않음 (pilot2의 '나눈 평가') |
| `all_*` | 시험 전체 / 8개 | 실제 상황처럼 전부 놓고 |
| `agnews_acc` | ag_news 일반 뉴스 4분류 | 원래 실력이 남았나 |

지표: `macro_f1`(라벨별 F1 평균, 불균형에 강함, 주 지표), `micro_f1`, `macro_ap`(임계값 없는 PR-AUC), `top1_acc`(1등 라벨이 정답 중 하나인 비율), `attack_recall`/`fpr`(1등이 BENIGN이 아니면 공격으로 봄), `separation` = recall - FPR

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
from datasets import load_dataset
from gliclass import GLiClassModel, ZeroShotClassificationPipeline

agnews = load_dataset('fancyzhx/ag_news', split='test').shuffle(seed=0).select(range(AGNEWS_N))
AG_LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

def subset(d, n, seed=0):
    return d if n is None or len(d) <= n else d.sample(n, random_state=seed)

val_df = subset(parts['val'], VAL_N).reset_index(drop=True)
test_df = subset(parts['test'], TEST_N).reset_index(drop=True)
is_held = test_df['labels'].apply(lambda ls: any(l in HELD for l in ls))
seen_df, held_df = test_df[~is_held].reset_index(drop=True), test_df[is_held].reset_index(drop=True)

def load_pipe(path):
    model = GLiClassModel.from_pretrained(path)
    tk = AutoTokenizer.from_pretrained(path)
    return ZeroShotClassificationPipeline(model, tk, classification_type='multi-label', device=DEVICE,
                                          max_length=MAX_LEN, progress_bar=False)

def score_matrix(pipe, texts, lids):
    out = pipe(list(texts), [text_of[l] for l in lids], threshold=0.0, batch_size=16)
    return np.array([[{x['label']: x['score'] for x in r}[text_of[l]] for l in lids] for r in out])

def metrics(df, S, lids, prefix):
    Y = np.array([[l in ls for l in lids] for ls in df['labels']], dtype=int)
    P = (S >= THRESHOLD).astype(int)
    p, r, f, _ = precision_recall_fscore_support(Y, P, average=None, zero_division=0)
    has = Y.sum(0) > 0                                     # 이 데이터에 정답이 있는 라벨만 평균
    ap = [average_precision_score(Y[:, j], S[:, j]) for j in range(len(lids)) if has[j]]
    mi = precision_recall_fscore_support(Y, P, average='micro', zero_division=0)
    top = np.array(lids)[S.argmax(1)]
    res = {f'{prefix}_macro_f1': f[has].mean(), f'{prefix}_micro_f1': mi[2],
           f'{prefix}_macro_p': p[has].mean(), f'{prefix}_macro_r': r[has].mean(),
           f'{prefix}_macro_ap': np.mean(ap),
           f'{prefix}_top1_acc': np.mean([t in ls for t, ls in zip(top, df['labels'])])}
    if 'BENIGN' in lids:
        att = df['labels'].apply(lambda ls: ls != ['BENIGN']).values
        pa = top != 'BENIGN'
        res[f'{prefix}_attack_recall'] = pa[att].mean() if att.any() else np.nan
        res[f'{prefix}_fpr'] = pa[~att].mean() if (~att).any() else np.nan
        res[f'{prefix}_separation'] = res[f'{prefix}_attack_recall'] - res[f'{prefix}_fpr']
    per = pd.DataFrame({'label': lids, 'n': Y.sum(0), 'precision': p, 'recall': r, 'f1': f, 'eval': prefix})
    return res, per

def evaluate(path, val_only=False):
    pipe = load_pipe(path)
    res, pers = {}, []
    groups = [('val', val_df, TRAIN)] if val_only else \
             [('val', val_df, TRAIN), ('seen', seen_df, TRAIN), ('held', held_df, HELD + ['BENIGN']), ('all', test_df, TRAIN + HELD)]
    for prefix, df, lids in groups:
        if len(df):
            r, per = metrics(df, score_matrix(pipe, df.text, lids), lids, prefix)
            res.update(r); pers.append(per)
    if not val_only:
        ag = pipe(agnews['text'], AG_LABELS, threshold=0.0, batch_size=16)
        res['agnews_acc'] = np.mean([AG_LABELS.index(max(a, key=lambda x: x['score'])['label']) == y
                                     for a, y in zip(ag, agnews['label'])])
    del pipe; torch.cuda.empty_cache()
    return res, (pd.concat(pers) if pers else None)

def show(r):
    return {k: round(float(v), 3) for k, v in r.items()}

print(f'평가 데이터: val {len(val_df)} | 시험 seen {len(seen_df)} / held {len(held_df)} / 전체 {len(test_df)} | ag_news {len(agnews)}')

## 6. 기준선 (파인튜닝 전 = k 0)

[하는 일] 원래 모델을 같은 지표로 잰다. 모든 비교의 출발점.

[볼 것] `held_*`(제로샷)와 `agnews_acc`가 파인튜닝 뒤에 이 값보다 크게 떨어지면 망각.

In [ ]:
t0 = time.time()
base, base_per = evaluate(MODEL)
base_per['k'], base_per['seed'] = 0, 0
print(f'기준선 {time.time() - t0:.0f}초')
show(base)

## 7. 학습 함수

[하는 일] GLiClass 공식 `train.py`를 그대로 쓰되 두 가지를 더한다.

1. **시드 고정**: `train.py`는 시드를 정하지 않아서 데이터 섞기와 dropout이 실행마다 달라진다. 작은 감싸개(`run_train.py`)가 시드를 고정한 뒤 `train.py`를 실행한다.
2. **에폭마다 체크포인트 + 검증 손실**: `save_steps = eval_steps = 1에폭 스텝 수`. 끝난 뒤 체크포인트마다 우리 지표로 검증해서 `SELECT_BY`가 가장 좋은 에폭을 고른다.

주의: `train.py`의 `type=bool` 인자(`--fp16` 등)는 `False`를 넣어도 참이 된다. 끄려면 아예 넘기지 않는다.

In [ ]:
WRAPPER = os.path.join(WORK, 'run_train.py')
with open(WRAPPER, 'w') as f:
    f.write('''import os, sys, random, runpy
import numpy as np, torch
seed = int(os.environ.get("TRAIN_SEED", "0"))
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
script = sys.argv[1]
sys.argv = sys.argv[1:]
runpy.run_path(script, run_name="__main__")
''')

ARCH = GLiClassModel.from_pretrained(MODEL).config.architecture_type

def train(train_file, seed, out_dir):
    n = len(json.load(open(train_file)))
    steps = max(1, math.ceil(n / BATCH))
    cmd = [sys.executable, WRAPPER, f'{REPO}/train.py', '--model_name', MODEL,
           '--data_path', train_file, '--val_data_path', f'{SPLIT}/val_gliclass.json',
           '--save_path', out_dir, '--problem_type', 'multi_label_classification', '--architecture_type', ARCH,
           '--encoder_lr', str(ENCODER_LR), '--others_lr', str(OTHERS_LR),
           '--num_epochs', str(EPOCHS), '--batch_size', str(BATCH), '--max_length', str(MAX_LEN),
           '--focal_loss_alpha', str(FOCAL_ALPHA), '--focal_loss_gamma', str(FOCAL_GAMMA),
           '--save_steps', str(steps), '--eval_steps', str(steps), '--save_total_limit', str(EPOCHS + 1),
           '--save_only_model', '--max_eval_samples', str(TRAINER_EVAL_N),
           '--logging_steps', '1', '--num_workers', '0', *EXTRA_ARGS]
    proc = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, 'TRAIN_SEED': str(seed)})
    if proc.returncode != 0:
        print(proc.stdout[-2000:], proc.stderr[-3000:])
        raise RuntimeError(f'학습 실패: {out_dir}')
    ckpts = sorted(glob.glob(f'{out_dir}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    log = json.load(open(f'{ckpts[-1]}/trainer_state.json'))['log_history']
    return ckpts, steps, log

def epoch_curve(log, steps):
    """trainer_state.json 기록 → 에폭마다 평균 학습 손실과 검증 손실."""
    tr = pd.DataFrame([x for x in log if 'loss' in x])
    ev = pd.DataFrame([x for x in log if 'eval_loss' in x])
    tr['epoch_i'] = np.ceil(tr.step / steps).astype(int)
    ev['epoch_i'] = (ev.step / steps).round().astype(int)
    return tr.groupby('epoch_i').loss.mean().rename('train_loss').to_frame().join(ev.set_index('epoch_i').eval_loss, how='outer')

## 8. 실험 실행 (k × seed)

[하는 일] k-shot 파일마다 학습 → 에폭별 검증 → 가장 좋은 에폭으로 시험 → 저장. 끝난 실행은 `runs.csv`에 남아 있으면 건너뛰므로 중간에 끊겨도 다시 돌리면 이어서 한다.

[볼 것] 실행마다 한 줄: 고른 에폭, 검증 F1, 시험 seen/held F1, ag_news.

[확인 질문] 고른 에폭(`best_epoch`)이 항상 마지막 에폭이면 무엇을 늘려봐야 할까? 항상 1에폭이면?

In [ ]:
RUNS, CURVES, PER = f'{OUT}/runs.csv', f'{OUT}/curves.csv', f'{OUT}/per_label.csv'
runs = pd.read_csv(RUNS) if os.path.exists(RUNS) else pd.DataFrame([{'k': 0, 'seed': 0, 'best_epoch': 0, **base}])
curves = pd.read_csv(CURVES) if os.path.exists(CURVES) else pd.DataFrame()
per_label = pd.read_csv(PER) if os.path.exists(PER) else base_per
done = set(zip(runs.k, runs.seed))
setting = dict(epochs=EPOCHS, encoder_lr=ENCODER_LR, others_lr=OTHERS_LR, batch=BATCH, extra=' '.join(EXTRA_ARGS))

for k in K_LIST:
    for seed in SEEDS:
        if (k, seed) in done:
            continue
        if (k, seed) not in train_files:
            print(f'k={k} seed={seed}: 학습 파일 없음 (04에서 라벨이 모자라 안 만듦)'); continue
        t0 = time.time()
        out_dir = os.path.join(WORK, 'runs', f'{VARIANT}_k{k}_s{seed}')
        shutil.rmtree(out_dir, ignore_errors=True)
        ckpts, steps, log = train(train_files[(k, seed)], seed, out_dir)

        cur = epoch_curve(log, steps)
        for ep, ck in enumerate(ckpts, start=1):            # 에폭마다 우리 지표로 검증
            r, _ = evaluate(ck, val_only=True)
            for m, v in r.items():
                cur.loc[ep, m] = v
        best_ep = int(cur[SELECT_BY].idxmax())
        m, per = evaluate(ckpts[best_ep - 1])                # 시험은 고른 에폭으로 한 번만

        if KEEP_MODELS:
            shutil.copytree(ckpts[best_ep - 1], os.path.join(WORK, 'best', f'{VARIANT}_k{k}_s{seed}'), dirs_exist_ok=True)
        shutil.rmtree(out_dir, ignore_errors=True)

        runs = pd.concat([runs, pd.DataFrame([{'k': k, 'seed': seed, 'best_epoch': best_ep, **setting, **m}])], ignore_index=True)
        curves = pd.concat([curves, cur.rename_axis('epoch').reset_index().assign(k=k, seed=seed)], ignore_index=True)
        per_label = pd.concat([per_label, per.assign(k=k, seed=seed)], ignore_index=True)
        runs.to_csv(RUNS, index=False); curves.to_csv(CURVES, index=False); per_label.to_csv(PER, index=False)
        print(f'k={k:2d} seed={seed} ({time.time() - t0:.0f}초) 에폭 {best_ep}/{EPOCHS} | '
              f'val F1 {m["val_macro_f1"]:.3f} | seen F1 {m["seen_macro_f1"]:.3f} | held F1 {m["held_macro_f1"]:.3f} | '
              f'판별력 {m["seen_separation"]:.3f} | ag_news {m["agnews_acc"]:.3f}')

## 9. 요약표

[하는 일] k마다 seed 평균 ± 표준편차. k = 0은 기준선.

[볼 것] seed끼리 표준편차가 평균 차이보다 크면 그 차이는 믿기 어렵다.

In [ ]:
COLS = ['best_epoch', 'val_macro_f1', 'seen_macro_f1', 'seen_macro_ap', 'seen_separation', 'seen_fpr',
        'held_macro_f1', 'held_macro_ap', 'all_macro_f1', 'agnews_acc']
summary = runs.groupby('k')[COLS].agg(['mean', 'std']).round(3)
summary

## 10. 그래프

1. **에폭별 손실**: 학습 손실은 내려가는데 검증 손실이 올라가는 지점 = 과적합 시작
2. **에폭별 검증 F1**: 고른 에폭이 어디인지
3. **k에 따른 성능**: k = 0(기준선)부터. 학습 전술(seen)은 오르고, 숨긴 전술(held)과 ag_news는 유지되는 게 목표
4. **라벨별 F1**: 어느 전술이 약한지

In [ ]:
import matplotlib.pyplot as plt

ks = sorted(curves.k.unique()) if len(curves) else []
if ks:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for k in ks:
        g = curves[curves.k == k].groupby('epoch')[['train_loss', 'eval_loss', 'val_macro_f1']].mean()
        axes[0].plot(g.index, g.train_loss, marker='o', label=f'k={k} train')
        axes[0].plot(g.index, g.eval_loss, ls='--', marker='s', label=f'k={k} val')
        axes[1].plot(g.index, g.val_macro_f1, marker='o', label=f'k={k}')
    axes[1].axhline(base['val_macro_f1'], color='grey', ls=':', label='before')
    axes[0].set(xlabel='epoch', ylabel='loss (focal)', title='1. loss per epoch (seed mean)')
    axes[1].set(xlabel='epoch', ylabel='val macro F1', title='2. validation macro F1 per epoch', ylim=(0, 1))
    for ax in axes: ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f'{OUT}/fig_epochs.png', dpi=200)

m, s = runs.groupby('k')[COLS].mean(), runs.groupby('k')[COLS].std().fillna(0)
fig, ax = plt.subplots(figsize=(7, 4))
for c, n in {'seen_macro_f1': 'seen tactics macro F1', 'held_macro_f1': 'held-out tactics macro F1 (zero-shot)',
             'seen_separation': 'attack recall - FPR', 'agnews_acc': 'ag_news acc (original skill)'}.items():
    ax.errorbar(m.index, m[c], yerr=s[c], marker='o', capsize=3, label=n)
ax.set(xlabel='examples per label (k), 0 = before fine-tuning', ylabel='score', xticks=list(m.index),
       ylim=(-0.3, 1.05), title=f'3. GLiClass edge few-shot fine-tuning ({VARIANT})')
ax.axhline(0, color='grey', lw=.8); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/fig_kshot.png', dpi=200)

h = per_label[per_label['eval'].isin(['seen', 'held'])].groupby(['label', 'k']).f1.mean().unstack('k')
h.index = [name.get(l, l) for l in h.index]
fig, ax = plt.subplots(figsize=(1.2 * h.shape[1] + 3, 0.5 * h.shape[0] + 1.5))
im = ax.imshow(h.values, vmin=0, vmax=1, cmap='viridis', aspect='auto')
ax.set(xticks=range(h.shape[1]), xticklabels=h.columns, yticks=range(h.shape[0]), yticklabels=h.index,
       xlabel='k', title='4. test F1 per label (seed mean)')
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        if not np.isnan(h.values[i, j]):
            ax.text(j, i, f'{h.values[i, j]:.2f}', ha='center', va='center', color='w' if h.values[i, j] < .6 else 'k', fontsize=8)
fig.colorbar(im); fig.tight_layout(); fig.savefig(f'{OUT}/fig_per_label.png', dpi=200)
print('저장:', sorted(f for f in os.listdir(OUT)))

## 11. 결과 읽는 법

| 관찰 | 뜻 | 다음에 해볼 것 |
|---|---|---|
| seen F1 오름, held F1·ag_news 유지 | 가설 방향과 일치 | 시드·k 늘려 확인 |
| seen F1 오름, held F1 하락 | 학습한 전술만 외움 | 학습률 낮추기, 에폭 줄이기 |
| ag_news 크게 하락 | 원래 실력 망각 | `EXTRA_ARGS`에 EWC (`--use_ewc --prev_data_path ...`) 또는 일반 데이터 섞기 |
| 학습 손실은 줄고 검증 손실은 오름 | 과적합 | 고른 에폭이 이른지 확인, 학습률 낮추기 |
| 학습 손실도 안 줄어듦 | 학습량 부족 | `ENCODER_LR`·`OTHERS_LR`을 1e-4로, `EPOCHS` 늘리기 (pilot2와 같은 점검) |
| best_epoch이 늘 마지막 | 더 학습할 여지 | `EPOCHS` 늘리기 |
| FPR이 높음 | 정상을 공격으로 많이 잡음 | `THRESHOLD`, BENIGN 예시 수 확인 |

학습률·에폭을 바꿔 비교할 때는 `OUT` 폴더 이름을 바꾸거나 `runs.csv`를 옮긴 뒤 다시 돌린다 (같은 k, seed는 건너뛰기 때문).
`runs.csv`, `curves.csv`, `per_label.csv`, `fig_*.png`를 회의에 가져가면 된다.